# Does Equity Plan Comprehensiveness Predict Black Poverty & Rent Burden?
## Regression Analysis — 20 Largest U.S. Cities (April 2026)

**Research question:** Do cities with stronger, more comprehensive racial equity plans have lower Black poverty rates and lower Black rent burden — or are plan quality scores unrelated to these outcomes once other city characteristics are controlled for?

**Dataset:** `racial_equity_20_cities.csv` — 20 largest U.S. cities by population  
**Source:** U.S. Census ACS 2023 + city equity reports + NYC REP April 2026  
**GitHub:** https://github.com/Trulove111/racial-equity-dataset  

---

### ⚠️ Statistical Caution
With **n = 20**, all results should be interpreted with care:
- Standard errors are large; 95% CIs will be wide
- Adding covariates quickly exhausts degrees of freedom
- No causal claims can be made — equity plans are endogenous (cities with worse outcomes may *adopt* stronger plans in response)
- Results are exploratory and hypothesis-generating, not confirmatory

We use OLS regression, robust (HC3) standard errors, bootstrapped CIs, and leverage/influence diagnostics throughout.

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
from statsmodels.stats.diagnostic import het_breuschpagan, linear_harvey_collier

# Install if needed
try:
    from sklearn.utils import resample
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'])
    from sklearn.utils import resample

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0D1B2A',
    'axes.facecolor':   '#111E2D',
    'axes.edgecolor':   '#2A3A4A',
    'axes.labelcolor':  '#C8D6E5',
    'axes.titlecolor':  '#F5F5F5',
    'xtick.color':      '#8899AA',
    'ytick.color':      '#8899AA',
    'grid.color':       '#1E2E3E',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'text.color':       '#C8D6E5',
    'font.family':      'sans-serif',
    'font.size':        11,
    'figure.dpi':       120,
})

AMBER   = '#F5A623'
TEAL    = '#2AABB8'
PINK    = '#D163A7'
GREEN   = '#27AE60'
PURPLE  = '#9B59B6'
ORANGE  = '#E67E22'
RED     = '#E74C3C'
BLUE    = '#3498DB'

STATUS_COLORS = {
    'Comprehensive': GREEN,
    'Partial':       AMBER,
    'Suspended':     PURPLE,
    'None':          RED,
}

print('Libraries loaded ✓')

## 1. Load & Prepare Data

In [ ]:
# ── Load CSV ──────────────────────────────────────────────────────────────────
# Adjust path as needed if running locally
try:
    df = pd.read_csv('racial_equity_20_cities.csv')
except FileNotFoundError:
    import urllib.request
    url = ('https://raw.githubusercontent.com/Trulove111/racial-equity-dataset'
           '/main/data/racial_equity_20_cities.csv')
    urllib.request.urlretrieve(url, 'racial_equity_20_cities.csv')
    df = pd.read_csv('racial_equity_20_cities.csv')

print(f'Loaded {len(df)} cities, {len(df.columns)} columns')
df.head(3)

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────

# Numeric equity score (already 0-100)
df['equity_score'] = pd.to_numeric(df['Equity Score (0-100)'], errors='coerce')

# Ordinal plan status (0=None, 1=Suspended, 2=Partial, 3=Comprehensive)
status_ord = {'None': 0, 'Suspended': 1, 'Partial': 2, 'Comprehensive': 3}
df['plan_ordinal'] = df['Plan Status'].map(status_ord)

# Dummy: 1 if Comprehensive, 0 otherwise
df['is_comprehensive'] = (df['Plan Status'] == 'Comprehensive').astype(int)

# Dummy: 1 if any active plan (Comprehensive or Partial)
df['has_active_plan'] = df['Plan Status'].isin(['Comprehensive', 'Partial']).astype(int)

# Accountability sub-scores (count of True flags out of 7)
bool_cols = ['Has Equity Plan','Measurable Goals','Has Timelines',
             'Has Budget Commitment','Has Enforcement','Equity Office Exists','Annual Reporting']
for c in bool_cols:
    df[c] = df[c].map({'true': 1, 'false': 0})
df['accountability_count'] = df[bool_cols].sum(axis=1)

# Core numeric outcome variables
num_cols = [
    'Black Poverty Rate (%)','White Poverty Rate (%)',
    'Black Rent Burden (%)','White Rent Burden (%)',
    'Black Median Income ($)','White Median Income ($)',
    'Median Household Income ($)',
    'Black Wage Gap (cents per White $)',
    'Black Unemployment Rate (%)','White Unemployment Rate (%)',
    'Commute Gap (min/week)',
    'Rent Burden Gap (pp Black-White)',
    'Unemployment Gap (ratio)',
    'Black Homeownership (%)','White Homeownership (%)',
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Poverty gap (Black − White)
df['poverty_gap'] = df['Black Poverty Rate (%)'] - df['White Poverty Rate (%)']

# Log income (reduces right-skew)
df['log_black_income'] = np.log(df['Black Median Income ($)'])
df['log_median_income'] = np.log(df['Median Household Income ($)'])

print('Features engineered:')
print(df[['City','Plan Status','equity_score','plan_ordinal','accountability_count']].to_string(index=False))

## 2. Exploratory Data Analysis

In [ ]:
# ── Summary stats by plan status ─────────────────────────────────────────────
summary = df.groupby('Plan Status').agg(
    n=('City', 'count'),
    avg_equity_score=('equity_score', 'mean'),
    avg_black_poverty=('Black Poverty Rate (%)', 'mean'),
    avg_black_rent_burden=('Black Rent Burden (%)', 'mean'),
    avg_poverty_gap=('poverty_gap', 'mean'),
    avg_rent_gap=('Rent Burden Gap (pp Black-White)', 'mean'),
).round(2)

# Reorder by accountability level
summary = summary.reindex(['Comprehensive','Partial','Suspended','None'])
print('\n── Summary by Plan Status ──')
print(summary.to_string())

In [ ]:
# ── Correlation matrix of key variables ──────────────────────────────────────
corr_vars = [
    'equity_score', 'plan_ordinal', 'accountability_count',
    'Black Poverty Rate (%)','White Poverty Rate (%)', 'poverty_gap',
    'Black Rent Burden (%)','Rent Burden Gap (pp Black-White)',
    'Black Wage Gap (cents per White $)',
    'Black Unemployment Rate (%)','Commute Gap (min/week)',
    'log_median_income',
]
corr_labels = [
    'Equity Score','Plan Ordinal','Acct. Count',
    'Black Poverty %','White Poverty %','Poverty Gap',
    'Black Rent Burden','Rent Burden Gap',
    'Wage Gap (¢/$)',
    'Black Unemployment','Commute Gap',
    'Log Median Income',
]

corr = df[corr_vars].corr()
corr.columns = corr_labels
corr.index = corr_labels

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 8},
    linewidths=0.5, linecolor='#0D1B2A',
    cbar_kws={'shrink': 0.7, 'label': 'Pearson r'},
)
ax.set_title('Correlation Matrix — Equity Plan Variables × Racial Disparity Outcomes\n'
             '(n=20 cities; lower triangle)', fontsize=12, pad=14)
plt.tight_layout()
plt.savefig('fig1_correlation_matrix.png', bbox_inches='tight', facecolor='#0D1B2A')
plt.show()
print('Figure 1 saved.')

In [ ]:
# ── Scatter: Equity Score vs. both outcomes ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (y_col, y_label) in zip(axes, [
    ('Black Poverty Rate (%)',  'Black Poverty Rate (%)'),
    ('Black Rent Burden (%)',   'Black Rent Burden (%)'),
]):
    for _, row in df.iterrows():
        color = STATUS_COLORS[row['Plan Status']]
        ax.scatter(row['equity_score'], row[y_col],
                   color=color, s=80, zorder=3, edgecolors='white', linewidths=0.4)
        ax.annotate(row['City'].split()[0],
                    (row['equity_score'], row[y_col]),
                    textcoords='offset points', xytext=(5, 3),
                    fontsize=7.5, color='#A0B4C8')

    # OLS fit line
    x = df['equity_score'].values
    y = df[y_col].values
    m, b, r, p, se = stats.linregress(x, y)
    xline = np.linspace(x.min()-2, x.max()+2, 100)
    ax.plot(xline, m*xline + b, color=TEAL, lw=2, ls='--', alpha=0.85,
            label=f'OLS fit  r={r:.2f}  p={p:.3f}')

    ax.set_xlabel('Equity Score (0–100)', labelpad=8)
    ax.set_ylabel(y_label, labelpad=8)
    ax.set_title(f'Equity Score vs. {y_label}', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Legend for plan status
handles = [mpatches.Patch(color=c, label=s) for s, c in STATUS_COLORS.items()]
fig.legend(handles=handles, title='Plan Status', loc='lower center',
           ncol=4, bbox_to_anchor=(0.5, -0.05), fontsize=9)

fig.suptitle('Equity Score vs. Black Poverty & Rent Burden — 20 Largest U.S. Cities',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig2_scatter_overview.png', bbox_inches='tight', facecolor='#0D1B2A')
plt.show()
print('Figure 2 saved.')

In [ ]:
# ── Box plots: outcomes by plan status ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

order = ['Comprehensive', 'Partial', 'Suspended', 'None']
palette = [STATUS_COLORS[s] for s in order]

for ax, (col, label) in zip(axes, [
    ('Black Poverty Rate (%)', 'Black Poverty Rate (%)'),
    ('Black Rent Burden (%)',  'Black Rent Burden (%)'),
]):
    sns.boxplot(data=df, x='Plan Status', y=col, order=order,
                palette=palette, ax=ax, width=0.5, linewidth=1.2,
                flierprops=dict(marker='o', markersize=5, markerfacecolor=AMBER))
    sns.stripplot(data=df, x='Plan Status', y=col, order=order,
                  color='white', size=5, alpha=0.7, ax=ax, jitter=0.12)

    # Annotate n per group
    for i, s in enumerate(order):
        n = (df['Plan Status'] == s).sum()
        ax.text(i, ax.get_ylim()[0] - 0.5, f'n={n}',
                ha='center', va='top', fontsize=8, color='#8899AA')

    ax.set_xlabel('Plan Status', labelpad=8)
    ax.set_ylabel(label, labelpad=8)
    ax.set_title(f'{label} by Plan Status', fontsize=11)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Outcome Distribution by Plan Status\n'
             '(white dots = individual cities)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('fig3_boxplots.png', bbox_inches='tight', facecolor='#0D1B2A')
plt.show()
print('Figure 3 saved.')

## 3. Regression Models

We test **three predictor specifications** for each outcome:

| Model | Predictor | Rationale |
|-------|-----------|----------|
| M1 | `equity_score` (continuous 0–100) | Tests linear relationship with composite score |
| M2 | `plan_ordinal` (0–3 ordinal) | Tests whether plan tier matters independently of score |
| M3 | `equity_score` + `log_median_income` | Partial out city wealth as a confounder |

All models use **HC3 heteroskedasticity-robust standard errors** (White sandwich estimator), which is appropriate when n is small and residuals may be non-constant.

In [ ]:
# ── Helper: fit OLS with HC3 robust SEs and print summary ─────────────────────
def fit_model(formula, data, label=''):
    model = smf.ols(formula, data=data).fit(cov_type='HC3')
    print(f'\n{"─"*60}')
    print(f'  {label}')
    print(f'  Formula: {formula}')
    print(f'  n={model.nobs:.0f}  R²={model.rsquared:.3f}  Adj.R²={model.rsquared_adj:.3f}')
    print(f'  F({model.df_model:.0f},{model.df_resid:.0f})={model.fvalue:.2f}  p(F)={model.f_pvalue:.4f}')
    print(model.summary2().tables[1].to_string())
    return model

# Rename columns for formula API (no special chars)
df2 = df.copy()
df2.columns = [c.replace('(','').replace(')','').replace('%','pct')
                .replace('/','_').replace(' ','_').replace('-','_')
                .replace('$','USD').strip('_')
                for c in df2.columns]

# ── OUTCOME 1: Black Poverty Rate ─────────────────────────────────────────────
print('\n' + '═'*60)
print('  OUTCOME 1: Black Poverty Rate (%)')
print('═'*60)

m1a = fit_model('Black_Poverty_Rate_pct ~ equity_score',
                df2, 'M1a — Poverty ~ Equity Score')

m1b = fit_model('Black_Poverty_Rate_pct ~ plan_ordinal',
                df2, 'M1b — Poverty ~ Plan Ordinal')

m1c = fit_model('Black_Poverty_Rate_pct ~ equity_score + log_median_income',
                df2, 'M1c — Poverty ~ Equity Score + Log Median Income (confounder)')

In [ ]:
# ── OUTCOME 2: Black Rent Burden ──────────────────────────────────────────────
print('\n' + '═'*60)
print('  OUTCOME 2: Black Rent Burden (%)')
print('═'*60)

m2a = fit_model('Black_Rent_Burden_pct ~ equity_score',
                df2, 'M2a — Rent Burden ~ Equity Score')

m2b = fit_model('Black_Rent_Burden_pct ~ plan_ordinal',
                df2, 'M2b — Rent Burden ~ Plan Ordinal')

m2c = fit_model('Black_Rent_Burden_pct ~ equity_score + log_median_income',
                df2, 'M2c — Rent Burden ~ Equity Score + Log Median Income')

In [ ]:
# ── OUTCOME 3: Gap measures (disparity, not level) ────────────────────────────
print('\n' + '═'*60)
print('  OUTCOME 3: Poverty Gap (Black − White pp)')
print('═'*60)
m3a = fit_model('poverty_gap ~ equity_score', df2, 'M3a — Poverty Gap ~ Equity Score')

print('\n' + '═'*60)
print('  OUTCOME 4: Rent Burden Gap (Black − White pp)')
print('═'*60)
m3b = fit_model('Rent_Burden_Gap_ppBlack_White ~ equity_score',
                df2, 'M3b — Rent Gap ~ Equity Score')

## 4. Model Diagnostics

In [ ]:
# ── Residual diagnostics for primary models ───────────────────────────────────
models_to_check = [
    (m1a, 'M1a: Poverty ~ Equity Score'),
    (m2a, 'M2a: Rent Burden ~ Equity Score'),
    (m1c, 'M1c: Poverty ~ Score + Income'),
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for row_i, (model, label) in enumerate(models_to_check):
    fitted = model.fittedvalues
    resid  = model.resid
    std_resid = resid / resid.std()

    # Panel 1: Residuals vs. Fitted
    ax = axes[row_i, 0]
    ax.scatter(fitted, resid, color=TEAL, s=60, alpha=0.8, edgecolors='white', lw=0.3)
    ax.axhline(0, color=AMBER, lw=1.5, ls='--')
    for i, name in enumerate(df['City'].str.split().str[0]):
        ax.annotate(name, (fitted.iloc[i], resid.iloc[i]),
                    fontsize=6.5, color='#7A8FA0', xytext=(3, 2),
                    textcoords='offset points')
    ax.set_xlabel('Fitted values')
    ax.set_ylabel('Residuals')
    ax.set_title(f'{label}\nResiduals vs. Fitted', fontsize=9)
    ax.grid(True, alpha=0.3)

    # Panel 2: Q-Q plot
    ax = axes[row_i, 1]
    qq = stats.probplot(resid, dist='norm')
    theoretical_q, sample_q = qq[0]
    ax.scatter(theoretical_q, sample_q, color=PINK, s=55, alpha=0.85, edgecolors='white', lw=0.3)
    fit_line = np.poly1d(np.polyfit(theoretical_q, sample_q, 1))
    xr = np.linspace(theoretical_q.min(), theoretical_q.max(), 100)
    ax.plot(xr, fit_line(xr), color=AMBER, lw=1.5, ls='--')
    ax.set_xlabel('Theoretical quantiles')
    ax.set_ylabel('Sample quantiles')
    ax.set_title('Normal Q-Q Plot', fontsize=9)
    ax.grid(True, alpha=0.3)

    # Panel 3: Cook's Distance
    ax = axes[row_i, 2]
    influence = OLSInfluence(model)
    cooks_d = influence.cooks_distance[0]
    threshold = 4 / len(df)
    colors = [RED if c > threshold else TEAL for c in cooks_d]
    ax.bar(range(len(cooks_d)), cooks_d, color=colors, alpha=0.85)
    ax.axhline(threshold, color=AMBER, lw=1.5, ls='--',
               label=f'Threshold 4/n={threshold:.3f}')
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df['City'].str.split().str[0], rotation=55, ha='right', fontsize=6.5)
    ax.set_ylabel("Cook's Distance")
    ax.set_title("Cook's Distance (red = influential)", fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Residual Diagnostics — Primary Models', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig4_diagnostics.png', bbox_inches='tight', facecolor='#0D1B2A')
plt.show()
print('Figure 4 saved.')

In [ ]:
# ── Breusch-Pagan heteroskedasticity test ─────────────────────────────────────
print('Breusch-Pagan Test for Heteroskedasticity (H₀: homoskedastic)\n')
for model, label in [
    (smf.ols('Black_Poverty_Rate_pct ~ equity_score', df2).fit(), 'M1a'),
    (smf.ols('Black_Rent_Burden_pct ~ equity_score', df2).fit(),  'M2a'),
]:
    bp_lm, bp_p, bp_f, bp_fp = het_breuschpagan(model.resid, model.model.exog)
    print(f'  {label}: LM={bp_lm:.3f}  p(LM)={bp_p:.4f}  '
          f'{"→ heteroskedastic (use robust SEs)" if bp_p < 0.1 else "→ no evidence of heteroskedasticity"}')

print('\n(HC3 robust standard errors used in all reported models regardless)')

In [ ]:
# ── VIF for the controlled model ─────────────────────────────────────────────
print('Variance Inflation Factors — M1c / M2c (equity_score + log_median_income)\n')
X_vif = df2[['equity_score','log_median_income']].dropna()
X_vif = sm.add_constant(X_vif)
for i, col in enumerate(X_vif.columns[1:], 1):
    vif = variance_inflation_factor(X_vif.values, i)
    print(f'  {col}: VIF = {vif:.3f}  {"(multicollinearity concern)" if vif > 5 else "(OK)"}')

## 5. Bootstrap Confidence Intervals

With n=20, asymptotic CIs from OLS are unreliable. We use **percentile bootstrap** (5,000 resamples) to get empirical CIs for the equity score slope coefficient.

In [ ]:
# ── Bootstrap slope CI ────────────────────────────────────────────────────────
N_BOOT = 5000
rng = np.random.default_rng(42)

def bootstrap_slope(x, y, n_boot=N_BOOT):
    slopes = []
    idx = np.arange(len(x))
    for _ in range(n_boot):
        sample = rng.choice(idx, size=len(idx), replace=True)
        xs, ys = x[sample], y[sample]
        if xs.std() == 0:
            continue
        m, _, _, _, _ = stats.linregress(xs, ys)
        slopes.append(m)
    slopes = np.array(slopes)
    return slopes.mean(), np.percentile(slopes, 2.5), np.percentile(slopes, 97.5), slopes

x_eq = df2['equity_score'].values

results = {}
for col, label in [
    ('Black_Poverty_Rate_pct',        'Black Poverty Rate'),
    ('Black_Rent_Burden_pct',         'Black Rent Burden'),
    ('poverty_gap',                   'Poverty Gap (B−W)'),
    ('Rent_Burden_Gap_ppBlack_White',  'Rent Burden Gap'),
]:
    y = df2[col].values
    mean_slope, ci_lo, ci_hi, boot_dist = bootstrap_slope(x_eq, y)
    r, p = stats.pearsonr(x_eq, y)
    results[col] = (label, mean_slope, ci_lo, ci_hi, r, p, boot_dist)
    sig = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.1 else 'ns'
    print(f'{label:30s}  slope={mean_slope:+.4f}  '
          f'95% CI [{ci_lo:+.4f}, {ci_hi:+.4f}]  r={r:.3f}  p={p:.4f} {sig}')

print('\nSignificance: *** p<.01  ** p<.05  * p<.10  ns=not significant')

In [ ]:
# ── Bootstrap slope distribution plots ───────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))

colors = [RED, ORANGE, PINK, AMBER]
for ax, (col, color) in zip(axes, zip(results.keys(), colors)):
    label, mean_s, ci_lo, ci_hi, r, p, boot_dist = results[col]
    ax.hist(boot_dist, bins=60, color=color, alpha=0.75, edgecolor='none')
    ax.axvline(mean_s, color='white', lw=2, label=f'Mean slope={mean_s:+.4f}')
    ax.axvline(ci_lo, color=TEAL, lw=1.5, ls='--', label=f'95% CI [{ci_lo:+.3f}, {ci_hi:+.3f}]')
    ax.axvline(ci_hi, color=TEAL, lw=1.5, ls='--')
    ax.axvline(0, color='#AAAAAA', lw=1, ls=':')
    ax.set_title(f'{label}\nr={r:.2f}  p={p:.3f}', fontsize=9)
    ax.set_xlabel('Bootstrap slope (per equity score point)', fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle('Bootstrap Slope Distributions — Equity Score Effect on Outcomes\n'
             '(5,000 resamples; dashed = 95% percentile CI; dotted = zero)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('fig5_bootstrap.png', bbox_inches='tight', facecolor='#0D1B2A')
plt.show()
print('Figure 5 saved.')

## 6. Coefficient Plot — All Models

In [ ]:
# ── Coefficient plot: equity_score across all models ──────────────────────────
all_models = [
    (m1a, 'M1a: Poverty ~ Score',            'Black Poverty'),
    (m1b, 'M1b: Poverty ~ Ordinal',          'Black Poverty'),
    (m1c, 'M1c: Poverty ~ Score + Income',   'Black Poverty'),
    (m2a, 'M2a: Rent ~ Score',               'Rent Burden'),
    (m2b, 'M2b: Rent ~ Ordinal',             'Rent Burden'),
    (m2c, 'M2c: Rent ~ Score + Income',      'Rent Burden'),
    (m3a, 'M3a: Poverty Gap ~ Score',        'Poverty Gap'),
    (m3b, 'M3b: Rent Gap ~ Score',           'Rent Gap'),
]

# Extract first non-intercept coefficient and its CI
coef_rows = []
for model, label, outcome in all_models:
    params = model.params
    ci = model.conf_int()
    # get first non-intercept
    pred = [p for p in params.index if p != 'Intercept'][0]
    coef_rows.append({
        'label': label, 'outcome': outcome,
        'coef':  params[pred],
        'ci_lo': ci.loc[pred, 0],
        'ci_hi': ci.loc[pred, 1],
        'p':     model.pvalues[pred],
        'predictor': pred,
    })

cdf = pd.DataFrame(coef_rows)

fig, ax = plt.subplots(figsize=(10, 7))
outcome_colors = {
    'Black Poverty': RED, 'Rent Burden': ORANGE,
    'Poverty Gap':   PINK, 'Rent Gap':    AMBER,
}

for i, row in cdf.iterrows():
    c = outcome_colors[row['outcome']]
    sig = row['p'] < 0.05
    ax.barh(i, row['coef'], color=c, alpha=0.85 if sig else 0.4, height=0.55)
    ax.errorbar(row['coef'], i,
                xerr=[[row['coef']-row['ci_lo']], [row['ci_hi']-row['coef']]],
                fmt='none', color='white', capsize=4, lw=1.5)
    sig_label = '●' if row['p'] < 0.05 else ('○' if row['p'] < 0.10 else '')
    ax.text(row['ci_hi'] + 0.002, i, f" {sig_label} p={row['p']:.3f}",
            va='center', fontsize=8, color='#A0B4C8')

ax.axvline(0, color='#AAAAAA', lw=1.2, ls='--')
ax.set_yticks(range(len(cdf)))
ax.set_yticklabels(cdf['label'], fontsize=9)
ax.set_xlabel('Coefficient (change in outcome per unit increase in predictor)', labelpad=8)
ax.set_title('Coefficient Plot — All Models\n'
             'Error bars = 95% CI (HC3 robust)  ● p<.05  ○ p<.10  dim = not significant',
             fontsize=10, pad=12)

handles = [mpatches.Patch(color=c, label=o) for o, c in outcome_colors.items()]
ax.legend(handles=handles, title='Outcome', fontsize=9, loc='lower right')
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('fig6_coef_plot.png', bbox_inches='tight', facecolor='#0D1B2A')
plt.show()
print('Figure 6 saved.')

## 7. Accountability Element Breakdown

Which specific plan elements (budget linkage, enforcement, timelines, etc.) correlate most strongly with lower Black poverty and rent burden?

In [ ]:
# ── Point-biserial correlations: each flag vs. outcomes ──────────────────────
flags = {
    'Has_Equity_Plan':    'Has Equity Plan',
    'Measurable_Goals':   'Measurable Goals',
    'Has_Timelines':      'Has Timelines',
    'Has_Budget_Commitment': 'Has Budget $',
    'Has_Enforcement':    'Has Enforcement',
    'Equity_Office_Exists': 'Equity Office',
    'Annual_Reporting':   'Annual Reporting',
}

flag_results = []
for col_raw, label in flags.items():
    col = col_raw  # already renamed in df2
    for outcome, out_label in [
        ('Black_Poverty_Rate_pct', 'Black Poverty %'),
        ('Black_Rent_Burden_pct',  'Black Rent Burden %'),
    ]:
        if col not in df2.columns:
            continue
        mask = df2[[col, outcome]].dropna()
        r, p = stats.pointbiserialr(mask[col], mask[outcome])
        flag_results.append({'Element': label, 'Outcome': out_label, 'r': r, 'p': p})

flag_df = pd.DataFrame(flag_results)
print('Point-Biserial Correlations: Accountability Flags vs. Outcomes\n')
print(flag_df.pivot(index='Element', columns='Outcome', values='r').round(3).to_string())

# Plot
pivot = flag_df.pivot(index='Element', columns='Outcome', values='r').round(3)
fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='barh', ax=ax, color=[RED, ORANGE], alpha=0.85, width=0.65)
ax.axvline(0, color='#AAAAAA', lw=1.2, ls='--')
ax.set_xlabel('Point-biserial correlation (r)', labelpad=8)
ax.set_title('Which Accountability Elements Correlate with Better Outcomes?\n'
             'Negative r = cities with this element have lower Black poverty/rent burden', fontsize=10)
ax.legend(title='Outcome', fontsize=9)
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('fig7_element_correlations.png', bbox_inches='tight', facecolor='#0D1B2A')
plt.show()
print('Figure 7 saved.')

## 8. Results Summary & Interpretation

In [ ]:
# ── Print formatted results table ─────────────────────────────────────────────
print('\n' + '═'*70)
print('  RESULTS SUMMARY')
print('═'*70)
print(f'{"Model":<40} {"Coeff":>8} {"p":>7} {"R²":>6} {"AdjR²":>7}')
print('─'*70)
for model, label, _ in all_models:
    params = model.params
    pred = [p for p in params.index if p != 'Intercept'][0]
    sig = '***' if model.pvalues[pred]<0.01 else '**' if model.pvalues[pred]<0.05 else '*' if model.pvalues[pred]<0.1 else '  '
    print(f'{label:<40} {params[pred]:>+8.4f} {model.pvalues[pred]:>6.4f}{sig} '
          f'{model.rsquared:>5.3f} {model.rsquared_adj:>6.3f}')
print('─'*70)
print('*** p<.01  ** p<.05  * p<.10')

## 9. Interpretation & Caveats

### What the regressions test
These models ask: **across these 20 cities at a single point in time (2023–2026), are cities with higher equity plan scores associated with lower Black poverty rates and lower Black rent burden?**

### Key interpretive cautions

1. **Reverse causality (endogeneity):** Cities adopt equity plans *because* they have severe racial disparities. A positive association between plan comprehensiveness and worse outcomes would be consistent with high-disparity cities trying harder — not plans causing harm. Negative or null associations are difficult to interpret without longitudinal data.

2. **Confounding:** City wealth (median income) is a powerful confounder. Wealthier cities have both lower Black poverty *and* more capacity to build institutional equity infrastructure. The controlled models (M1c, M2c) attempt to partial this out, but with n=20 the controlled estimates are fragile.

3. **Selection on policy adoption:** The 4 cities with *Comprehensive* plans (NYC, Chicago, Philadelphia, Seattle) are all large, high-cost, Democratic-governed cities with large Black populations. Any "Comprehensive" effect is partially a proxy for these structural characteristics.

4. **Plan age vs. outcome:** Plans adopt at different times. A city that passed a comprehensive plan in 2023 has had far less time to affect ACS 2023 outcomes than one that passed in 2017. This analysis cannot account for implementation lag.

5. **N=20 power:** At n=20, OLS has very low power to detect small effects. Null results do not mean no relationship exists — they may simply reflect insufficient data.

### How to use these results
- Use as **hypothesis generation** for a longitudinal study comparing the same cities pre/post plan adoption
- The accountability element correlations (Section 7) are potentially more actionable: they identify *which specific elements* (budget linkage, enforcement) have the strongest associations
- A difference-in-differences design comparing cities that adopted plans vs. those that didn't (or pre/post passage) would be far more credible causally

---
*Dataset v1.1.0 — April 2026 | GitHub: https://github.com/Trulove111/racial-equity-dataset*